Cropping the DEM tiles on all four sides to regulate the size of 1km*1km

In [ ]:
from pathlib import Path
import numpy as np
import tifffile
from shutil import copy2

# -------------------------------
# 1. FOLDERS
# -------------------------------

# Folder where your original DEM tiles live (102 x 102 px)
DEM_IN  = Path(r"D:\Sunita_Thesis\Datasets\ProjectedDEM_10\ProjectedDEM_10")

# New folder for cropped DEM tiles (100 x 100 px)
DEM_OUT = Path(r"D:\Sunita_Thesis\Datasets\ProjectedDEM_10\CROPPED_DEM_10")
DEM_OUT.mkdir(parents=True, exist_ok=True)

print("Input DEM folder :", DEM_IN)
print("Output DEM folder:", DEM_OUT)


# -------------------------------
# 2. SMALL HELPER FUNCTIONS
# -------------------------------

def read_dem_with_meta(path):
    """
    Open a DEM GeoTIFF and return:
      - data array
      - metadata needed to write a new file:
          * pixel size
          * upper-left world coordinate
          * CRS geokeys
          * NoData value
    """
    with tifffile.TiffFile(path) as tf:
        page = tf.pages[0]
        arr = page.asarray()
        tags = page.tags

        # --- pixel size: ModelPixelScaleTag (33550) ---
        scale_tag = tags.get(33550)
        if scale_tag is None:
            raise RuntimeError(f"No ModelPixelScaleTag (33550) in {path}")
        sx, sy = float(scale_tag.value[0]), float(scale_tag.value[1])

        # --- tiepoint: ModelTiepointTag (33922) ---
        tie_tag = tags.get(33922)
        if tie_tag is None:
            raise RuntimeError(f"No ModelTiepointTag (33922) in {path}")
        i0, j0, _, X0, Y0, _ = map(float, tie_tag.value[:6])

        # upper-left world coordinate of pixel (0,0)
        ulx = X0 + i0 * sx
        uly = Y0 - j0 * sy

        # --- CRS geokeys (projection info) ---
        gk_tag = tags.get(34735)  # GeoKeyDirectoryTag
        gd_tag = tags.get(34736)  # GeoDoubleParamsTag
        ga_tag = tags.get(34737)  # GeoAsciiParamsTag

        geokey   = np.array(gk_tag.value, dtype=np.uint16) if gk_tag else None
        geodbl   = np.array(gd_tag.value, dtype=np.float64) if gd_tag else None
        geoascii = ga_tag.value if ga_tag else None
        if isinstance(geoascii, bytes):
            geoascii = geoascii.decode("ascii", "ignore")

        # --- NoData value (GDAL_NODATA tag 42113) ---
        nd_tag = tags.get(42113)
        nodata = None
        if nd_tag is not None:
            v = nd_tag.value
            try:
                nodata = float(v.decode("ascii")) if isinstance(v, bytes) else float(str(v))
            except Exception:
                nodata = None

    meta = {
        "px": (abs(sx), abs(sy)),
        "ul": (ulx, uly),
        "geokey": geokey,
        "geodbl": geodbl,
        "geoascii": geoascii,
        "nodata": nodata,
    }
    return arr, meta


def write_dem(path_out, arr, meta):
    """
    Write a DEM GeoTIFF with:
      - given array
      - updated upper-left coordinate
      - same CRS + NoData
    """
    px_x, px_y = meta["px"]
    ulx, uly   = meta["ul"]
    geokey     = meta["geokey"]
    geodbl     = meta["geodbl"]
    geoascii   = meta["geoascii"]
    nodata     = meta["nodata"]

    extratags = []

    # ModelPixelScaleTag (33550): pixel size
    extratags.append(
        (33550, 'd', 3, (float(px_x), float(px_y), 0.0), False)
    )

    # ModelTiepointTag (33922): tie pixel (0,0,0) to world (ulx, uly, 0)
    extratags.append(
        (33922, 'd', 6, (0.0, 0.0, 0.0, float(ulx), float(uly), 0.0), False)
    )

    # CRS geokeys
    if geokey is not None:
        extratags.append((34735, 'H', geokey.size, geokey, False))
    if geodbl is not None:
        extratags.append((34736, 'd', geodbl.size, geodbl, False))
    if geoascii is not None:
        extratags.append((34737, 's', len(geoascii), geoascii, False))

    # GDAL_NODATA
    if nodata is not None:
        nd_str = str(nodata)
        extratags.append((42113, 's', len(nd_str), nd_str, False))

    tifffile.imwrite(
        path_out,
        arr,
        dtype=arr.dtype,
        photometric="minisblack",
        compression=None,   # uncompressed → avoids imagecodecs/LZW problems
        metadata=None,
        extratags=extratags,
    )


def crop_one_pixel(path_in, path_out):
    """
    Read a DEM tile, crop 1 pixel from each side (-> 100x100),
    shift the upper-left coordinate by 1 pixel,
    and save as new GeoTIFF.
    """
    arr, meta = read_dem_with_meta(path_in)

    # safety check
    if arr.shape[0] <= 2 or arr.shape[1] <= 2:
        print("  WARNING:", path_in.name, "is too small to crop, copying as is.")
        copy2(path_in, path_out)
        return "copied (too small)"

    # --- crop 1 pixel on all sides ---
    cropped = arr[1:-1, 1:-1]     # remove first/last row and column

    # --- update upper-left coordinate ---
    px_x, px_y = meta["px"]
    ulx_old, uly_old = meta["ul"]

    # move UL by +1 pixel in x (east) and -1 pixel in y (south/north-up)
    ulx_new = ulx_old + px_x
    uly_new = uly_old - px_y

    meta_new = meta.copy()
    meta_new["ul"] = (ulx_new, uly_new)

    # --- write out ---
    write_dem(path_out, cropped, meta_new)

    return f"cropped to {cropped.shape[1]}x{cropped.shape[0]} px"


# -------------------------------
# 3. RUN OVER ALL DEM TILES
# -------------------------------

dem_files = sorted(DEM_IN.glob("*.tif"))
print(f"Found {len(dem_files)} DEM tiles")

# You can test on a few first tiles by setting TEST_N to a small number
TEST_N = None     # change to None to process ALL tiles

if TEST_N is None:
    files_to_do = dem_files
else:
    files_to_do = dem_files[:TEST_N]

for i, src in enumerate(files_to_do, 1):
    dst = DEM_OUT / src.name
    status = crop_one_pixel(src, dst)
    print(f"[{i}/{len(files_to_do)}] {src.name} -> {status}")

print("Done.")


In [ ]:
from pathlib import Path
import numpy as np
import tifffile
import matplotlib.pyplot as plt

# ---------- SETTINGS ----------
DEM_OUT = Path(r"D:\Sunita_Thesis\Datasets\ProjectedDEM_10\CROPPED_DEM_10")
N_SHOW = 21  # how many cropped DEMs to inspect

dem_files = sorted(DEM_OUT.glob("*.tif"))
print(f"Found {len(dem_files)} cropped DEM tiles")
dem_files = dem_files[:N_SHOW]
print(f"Showing first {len(dem_files)}:\n")

# ---------- HELPER FUNCTIONS ----------

def read_dem_info(path):
    """Return array + basic geoinfo + nodata + EPSG if possible."""
    with tifffile.TiffFile(path) as tf:
        page = tf.pages[0]
        arr = page.asarray()
        tags = page.tags

        # Pixel size (ModelPixelScale, 33550)
        scale_tag = tags.get(33550)
        if scale_tag is not None:
            sx, sy = float(scale_tag.value[0]), float(scale_tag.value[1])
        else:
            sx = sy = None

        # Tiepoint (ModelTiepoint, 33922): (i, j, k, X, Y, Z)
        tie_tag = tags.get(33922)
        if tie_tag is not None:
            i0, j0, _, X0, Y0, _ = map(float, tie_tag.value[:6])
            ulx = X0 + i0 * sx if sx is not None else None
            uly = Y0 - j0 * sy if sy is not None else None
        else:
            ulx = uly = None

        # NoData (GDAL_NODATA, 42113)
        nd_tag = tags.get(42113)
        nodata = None
        if nd_tag is not None:
            v = nd_tag.value
            try:
                nodata = float(v.decode("ascii")) if isinstance(v, bytes) else float(str(v))
            except Exception:
                nodata = None

        # CRS / EPSG: from GeoKeyDirectory (34735)
        epsg = None
        gk_tag = tags.get(34735)
        if gk_tag is not None:
            # GeoKeyDirectory is an array of uint16
            gk = np.array(gk_tag.value, dtype=np.uint16)
            # first 4 values are header; then keys in groups of 4 (keyID, tiffTagLoc, count, valueOffset)
            if gk.size >= 4:
                num_keys = gk[3]
                key_entries = gk[4:].reshape(num_keys, 4)
                for key_id, tiff_loc, count, value_off in key_entries:
                    # ProjectedCSTypeGeoKey (3072) usually holds the EPSG code for projected CRS
                    if key_id == 3072 and tiff_loc == 0:
                        epsg = int(value_off)
                        break

    info = {
        "arr": arr,
        "px": (sx, sy),
        "ul": (ulx, uly),
        "nodata": nodata,
        "epsg": epsg,
    }
    return info

def norm_img(a, nodata=None):
    """Simple normalization for plotting, masking nodata."""
    a = a.astype("float32")
    if nodata is not None:
        a = np.where(a == nodata, np.nan, a)
    a[a <= -9999] = np.nan  # catch DEM nodata just in case
    lo, hi = np.nanpercentile(a, [2, 98])
    if not np.isfinite(lo): lo = np.nanmin(a)
    if not np.isfinite(hi): hi = np.nanmax(a)
    if hi <= lo: hi = lo + 1e-6
    return np.clip((a - lo) / (hi - lo), 0, 1)

# ---------- PRINT INFO + COLLECT FOR PLOTTING ----------

infos = []

for path in dem_files:
    info = read_dem_info(path)
    infos.append((path, info))

    arr = info["arr"]
    sx, sy = info["px"]
    ulx, uly = info["ul"]
    nodata = info["nodata"]
    epsg   = info["epsg"]

    print(f"{path.name}")
    print(f"  shape: {arr.shape}, dtype: {arr.dtype}")
    print(f"  pixel size: ({sx}, {sy})")
    print(f"  upper-left (X, Y): ({ulx}, {uly})")
    print(f"  NoData: {nodata}")
    print(f"  EPSG (if found): {epsg}\n")

# ---------- VISUALIZE THE 10 CROPPED DEMs ----------

n = len(infos)
n_rows = 2
n_cols = int(np.ceil(n / n_rows))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 4*n_rows))
axes = np.atleast_1d(axes).ravel()

for ax, (path, info) in zip(axes, infos):
    img = norm_img(info["arr"], info["nodata"])
    ax.imshow(img, cmap="terrain")
    ax.set_title(path.name, fontsize=8)
    ax.axis("off")

# hide any extra axes if there are fewer than n_rows*n_cols images
for ax in axes[n:]:
    ax.axis("off")

plt.suptitle("First cropped DEM tiles (100x100)", y=0.98)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
from pathlib import Path
import tifffile

DEM_OUT = Path(r"D:\Sunita_Thesis\Datasets\ProjectedDEM_10\CROPPED_DEM_10")

for path in sorted(list(DEM_OUT.glob("*.tif")))[:30]:  # check first 10
    arr = tifffile.imread(path)
    nodata_pixels = np.sum(arr == -9999)
    print(f"{path.name}: nodata pixels = {nodata_pixels}")
